# SOP 全量数据预处理与特征构建

本 Notebook 会读取整个 Stanford Online Products ZIP，并生成一个可供后续实验直接读取的处理后图片目录。

主要输出：
- images/train 或 images/test/class_id/image_id.jpg：通过原图检查后，转换为 224×224 RGB 的图片；
- clean_manifest.csv：保留图片、商品、大类、官方 split、原始尺寸及处理后图片路径；
- excluded_samples.csv、excluded_products.csv：记录被排除的图片和只剩单张合格图的商品；
- image_features_hsv_hog.npz：每张保留图片的 192 维颜色直方图 + 空间 HOG 特征；
- audit_summary.json：全量处理数量统计。

原图先按原始宽高检查，再做模型输入尺寸转换；缩放不会用于满足老师的数据规模条件。原始 ZIP 不会被修改。全程保留官方 train/test 标记，不创建新的实验划分。

本阶段不生成随机图片对、不做合成遮挡、不拟合标准化/PCA 参数、不训练和调参。之后的交叉验证必须按商品 class_id 分组；遮挡增强和从数据中估计的参数都要放在训练折内部。

In [ ]:
from pathlib import Path, PurePosixPath
import sys
from io import BytesIO
from zipfile import ZipFile
import json
import time

import numpy as np
import pandas as pd
from PIL import Image
import cv2

cwd = Path.cwd().resolve()
module_dir = next((p for p in [cwd, cwd / "notebooks", *cwd.parents] if (p / "sop_features.py").is_file()), None)
if module_dir is None:
    raise FileNotFoundError("请将 sop_features.py 与 Notebook 一起放在项目根目录或 notebooks 文件夹中。")
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from sop_features import image_feature_vector, resize_for_model

print("OpenCV:", cv2.__version__)
print("Pillow:", Image.__version__)

## 1. 数据路径和输出位置

代码从当前目录及父目录查找 Stanford_Online_Products.zip。若找不到，会提示输入完整路径。输出默认写到 Notebook 当前工作目录下的 processed_sop 文件夹。

为避免覆盖旧结果，如果 processed_sop 已存在且非空，代码会停止；请先在下方把 OUTPUT_DIR 改成一个新的输出文件夹名称。

In [ ]:
def find_sop_zip():
    for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        candidate = folder / "Stanford_Online_Products.zip"
        if candidate.is_file():
            return candidate
    entered = input("请输入 Stanford_Online_Products.zip 的完整路径：").strip().strip('"')
    candidate = Path(entered).expanduser()
    if not candidate.is_file():
        raise FileNotFoundError("找不到数据 ZIP: " + str(candidate))
    return candidate.resolve()

ZIP_PATH = find_sop_zip()
OUTPUT_DIR = Path.cwd().resolve() / "processed_sop"
MIN_ORIGINAL_SIDE = 100
PROCESSED_SIZE = 224

if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(
        "输出目录已存在且非空。为避免混入旧图片，请把 OUTPUT_DIR 改成新的目录名，例如 processed_sop_run2。"
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("数据 ZIP:", ZIP_PATH)
print("输出目录:", OUTPUT_DIR)

## 2. 读取官方索引并保留原始 train/test 标记

SOP 已提供官方 train/test；同一商品原则上不会跨两个部分。本作业仍需自己实现至少 10 折外层交叉验证和至少 3 折内层调参，所以不能只用一次官方 train/test 结果。建议后续在官方 train 商品上做按 SKU 分组的嵌套交叉验证，官方 test 留到最后作为额外检查，不参与调参。

In [ ]:
with ZipFile(ZIP_PATH, "r") as zf:
    names = set(zf.namelist())
    train_members = [n for n in names if PurePosixPath(n).name == "Ebay_train.txt"]
    test_members = [n for n in names if PurePosixPath(n).name == "Ebay_test.txt"]
    if len(train_members) != 1 or len(test_members) != 1:
        raise ValueError("ZIP 中应各有且仅有一个 Ebay_train.txt 和 Ebay_test.txt。")

    train_member, test_member = train_members[0], test_members[0]
    train_prefix = PurePosixPath(train_member).parent.as_posix()
    test_prefix = PurePosixPath(test_member).parent.as_posix()
    if train_prefix != test_prefix:
        raise ValueError("train/test 索引不在同一数据目录下。")
    ZIP_PREFIX = "" if train_prefix == "." else train_prefix + "/"

    train_df = pd.read_csv(BytesIO(zf.read(train_member)), sep=r"\s+", engine="python")
    test_df = pd.read_csv(BytesIO(zf.read(test_member)), sep=r"\s+", engine="python")

required_columns = ["image_id", "class_id", "super_class_id", "path"]
for split_name, frame in [("train", train_df), ("test", test_df)]:
    if list(frame.columns) != required_columns:
        raise ValueError(split_name + " 索引列名或顺序与预期不一致。")
    frame["source_split"] = split_name

index_df = pd.concat([train_df, test_df], ignore_index=True)
if index_df["image_id"].duplicated().any():
    raise ValueError("发现重复 image_id。")
if (index_df.groupby("class_id")["source_split"].nunique() > 1).any():
    raise ValueError("同一商品出现在官方 train/test 两侧，请停止并检查标签。")
if (index_df.groupby("class_id")["super_class_id"].nunique() > 1).any():
    raise ValueError("同一商品对应多个 super_class_id。")

print("索引图片数:", len(index_df))
print("商品数:", index_df["class_id"].nunique())
print(index_df.groupby("source_split").agg(
    images=("image_id", "size"), products=("class_id", "nunique")
))

## 3. 全量检查原图

逐张检查 ZIP 中图片是否存在、能否完整解码，以及原始宽和高是否均至少为 100。只有通过检查的图片才进入下一步。当前只读取并记录，不会解压或改写原图。

In [ ]:
audit_rows = []
start_time = time.time()

with ZipFile(ZIP_PATH, "r") as zf:
    for i, row in enumerate(index_df.itertuples(index=False), start=0):
        relative = PurePosixPath(str(row.path))
        if relative.is_absolute() or ".." in relative.parts:
            audit_rows.append({
                "image_id": int(row.image_id), "original_width": None,
                "original_height": None, "status": "excluded",
                "reason": "unsafe_metadata_path"
            })
            continue

        member = ZIP_PREFIX + relative.as_posix()
        if member not in names:
            audit_rows.append({
                "image_id": int(row.image_id), "original_width": None,
                "original_height": None, "status": "excluded",
                "reason": "missing_file_in_zip"
            })
            continue

        try:
            raw_bytes = zf.read(member)
            with Image.open(BytesIO(raw_bytes)) as opened:
                width, height = opened.size
                opened.load()

            if width < MIN_ORIGINAL_SIDE or height < MIN_ORIGINAL_SIDE:
                audit_rows.append({
                    "image_id": int(row.image_id), "original_width": width,
                    "original_height": height, "status": "excluded",
                    "reason": "original_smaller_than_100"
                })
            else:
                audit_rows.append({
                    "image_id": int(row.image_id), "original_width": width,
                    "original_height": height, "status": "eligible", "reason": ""
                })
        except Exception as exc:
            audit_rows.append({
                "image_id": int(row.image_id), "original_width": None,
                "original_height": None, "status": "excluded",
                "reason": "unreadable: " + type(exc).__name__
            })

        if (i + 1) % 5000 == 0:
            elapsed = round((time.time() - start_time) / 60, 1)
            print("已检查", i + 1, "/", len(index_df), "张；耗时", elapsed, "分钟")

audit_df = pd.DataFrame(audit_rows)
audit_df = audit_df.set_index("image_id").reindex(index_df["image_id"]).reset_index()
audit_df = pd.concat(
    [index_df.reset_index(drop=True), audit_df.drop(columns=["image_id"])],
    axis=1
)
print("原图检查完成；耗时分钟:", round((time.time() - start_time) / 60, 1))
print(audit_df["reason"].value_counts(dropna=False))

## 4. 排除只剩一张合格图片的商品

先完成坏图和原图尺寸筛选，再统计每个 class_id 剩余的合格图片数。仅剩 1 张图片的商品无法形成同商品正样本对，因此将该商品排除。

In [ ]:
eligible_mask = audit_df["status"].eq("eligible")
eligible_counts = audit_df.loc[eligible_mask, "class_id"].value_counts()
single_image_products = set(eligible_counts[eligible_counts == 1].index)

singleton_mask = eligible_mask & audit_df["class_id"].isin(single_image_products)
audit_df.loc[singleton_mask, "status"] = "excluded"
audit_df.loc[singleton_mask, "reason"] = "only_one_eligible_image_for_product"

clean_manifest = audit_df.loc[audit_df["status"].eq("eligible")].copy().reset_index(drop=True)
excluded_samples = audit_df.loc[audit_df["status"].eq("excluded")].copy().reset_index(drop=True)
excluded_products = (
    audit_df.loc[singleton_mask, ["class_id", "super_class_id", "source_split"]]
    .drop_duplicates()
    .sort_values("class_id")
    .reset_index(drop=True)
)

print("通过原图检查:", int(eligible_mask.sum()))
print("只剩一张合格图片的商品:", len(single_image_products))
print("清洁图片数:", len(clean_manifest))
print("清洁商品数:", clean_manifest["class_id"].nunique())
print(clean_manifest.groupby("source_split").agg(
    images=("image_id", "size"), products=("class_id", "nunique")
))

## 5. 生成处理后的图片文件夹和逐图特征

第二遍读取清洁图片，将每张图片保持纵横比缩放并填充为 224×224 RGB，然后保存到 images/train 或 images/test/class_id/image_id.jpg。原始尺寸已在第 3 步检查和记录；这次缩放只为后续模型统一输入，不用于满足数据集门槛。

同时计算 HSV+空间 HOG 特征。保存过程中每 5,000 张显示一次进度。

In [ ]:
processed_paths = []
feature_matrix = np.empty((len(clean_manifest), 192), dtype=np.float32)
start_time = time.time()

with ZipFile(ZIP_PATH, "r") as zf:
    for i, row in enumerate(clean_manifest.itertuples(index=False), start=0):
        relative = PurePosixPath(str(row.path))
        member = ZIP_PREFIX + relative.as_posix()
        raw_bytes = zf.read(member)

        with Image.open(BytesIO(raw_bytes)) as opened:
            opened.load()
            original_image = opened.convert("RGB")

        processed_image = resize_for_model(original_image, size=PROCESSED_SIZE)
        relative_output = (
            Path("images") / str(row.source_split) / str(row.class_id)
            / (str(row.image_id) + ".jpg")
        )
        output_path = OUTPUT_DIR / relative_output
        output_path.parent.mkdir(parents=True, exist_ok=True)
        processed_image.save(output_path, format="JPEG", quality=95, optimize=True)

        processed_paths.append(relative_output.as_posix())
        feature_matrix[i] = image_feature_vector(processed_image)

        if (i + 1) % 5000 == 0:
            elapsed = round((time.time() - start_time) / 60, 1)
            print("已生成", i + 1, "/", len(clean_manifest), "张；耗时", elapsed, "分钟")

clean_manifest["processed_relative_path"] = processed_paths
clean_manifest["processed_width"] = PROCESSED_SIZE
clean_manifest["processed_height"] = PROCESSED_SIZE
print("图片转换与特征计算完成；耗时分钟:", round((time.time() - start_time) / 60, 1))

## 6. 保存清单、特征和数量摘要

特征文件行顺序与 clean_manifest.csv 完全一致；同时保存 image_id、class_id、super_class_id 和官方 source_split，后续代码应按这些键关联，不能只假设数组顺序。

In [ ]:
feature_names = (
    ["hsv_h%02d" % i for i in range(16)]
    + ["hsv_s%02d" % i for i in range(16)]
    + ["hsv_v%02d" % i for i in range(16)]
    + ["hog_r%d_c%d_b%d" % (r, c, b)
       for r in range(4) for c in range(4) for b in range(9)]
)

clean_manifest.to_csv(
    OUTPUT_DIR / "clean_manifest.csv", index=False, encoding="utf-8-sig"
)
excluded_samples.to_csv(
    OUTPUT_DIR / "excluded_samples.csv", index=False, encoding="utf-8-sig"
)
excluded_products.to_csv(
    OUTPUT_DIR / "excluded_products.csv", index=False, encoding="utf-8-sig"
)
audit_df.to_csv(
    OUTPUT_DIR / "full_image_audit.csv", index=False, encoding="utf-8-sig"
)
np.savez_compressed(
    OUTPUT_DIR / "image_features_hsv_hog.npz",
    image_ids=clean_manifest["image_id"].to_numpy(dtype=np.int64),
    class_ids=clean_manifest["class_id"].to_numpy(dtype=np.int64),
    super_class_ids=clean_manifest["super_class_id"].to_numpy(dtype=np.int64),
    source_split=clean_manifest["source_split"].astype(str).to_numpy(),
    feature_names=np.asarray(feature_names, dtype=str),
    X=feature_matrix
)

summary = {
    "zip_path": str(ZIP_PATH),
    "indexed_images": int(len(index_df)),
    "indexed_products": int(index_df["class_id"].nunique()),
    "clean_images": int(len(clean_manifest)),
    "clean_products": int(clean_manifest["class_id"].nunique()),
    "images_by_official_split": {
        str(k): int(v) for k, v in clean_manifest["source_split"].value_counts().items()
    },
    "excluded_samples_by_reason": {
        str(k): int(v) for k, v in excluded_samples["reason"].value_counts().items()
    },
    "feature_dimension": int(feature_matrix.shape[1]),
    "processed_image_size": [PROCESSED_SIZE, PROCESSED_SIZE],
    "new_train_validation_split_created": False
}
(OUTPUT_DIR / "audit_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("完成。处理后的图片文件夹:", (OUTPUT_DIR / "images").resolve())

## 后续实验如何读取

- 读取 clean_manifest.csv，按 processed_relative_path 找到处理后的图片；class_id 是商品标签，source_split 是官方 train/test 标记。
- 读取 image_features_hsv_hog.npz，X 是 192 维逐图特征。
- 单独实验代码可从同目录的 sop_features.py 导入 resize_for_model、image_feature_vector、sift_pair_features。
- 主评估按 class_id/SKU 分组做至少 10 折外层交叉验证；内层至少 3 折调参。只在训练折内生成增强样本和拟合数据相关的标准化/降维参数。官方 test 不参与调参。
- 这个文件夹内图片是派生实验输入，不要放进最终作业提交 ZIP；老师要求不提交数据。